# IberoamericaBooks — workflow ETL paso a paso

> **Portfolio / Demo Version**

Este notebook reproduce de forma didáctica la estructura del workflow original: creación de la base de datos, lectura de fuentes heterogéneas, normalización, validación, deduplicación, enriquecimiento, persistencia y consultas.

La demo utiliza metadatos bibliográficos públicos y variables comerciales semisintéticas generadas de forma determinista. No contiene registros privados, credenciales ni claves de API.

## Objetivo y arquitectura

El problema original consistía en integrar catálogos editoriales y métricas procedentes de instituciones con estructuras incompatibles. El resultado debía conservar la procedencia de cada fila y ofrecer un modelo relacional consultable.

```text
SIMEH ──────────────┐
Plantilla propia ───┼─> Ingesta ─> Normalización ─> Validación ─> Deduplicación
SciELO ─────────────┘                                      │
                                                          v
Ventas + OpenAlex + Altmetric ──────────> Enriquecimiento ─> SQLite
WorldCat ───────────────────────────────> Prototipo documentado
```

La interfaz Streamlit y este notebook llaman al mismo paquete de Python situado en `src/iberoamerica_books/`.

## 0) Preparación del entorno

La ruta se calcula de forma relativa para que el notebook funcione después de clonar el repositorio, sin depender del ordenador del autor.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3
import sys

import pandas as pd
try:
    from IPython.display import display
except ImportError:  # Permite validar las celdas también fuera de Jupyter.
    display = print

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from iberoamerica_books.cleaning import clean_isbn, is_valid_isbn13, normalize_text
from iberoamerica_books.database import SCHEMA, connect, read_only_query
from iberoamerica_books.pipeline import (
    SOURCE_SPECS, _canonical_catalogue, _read_sources, run_pipeline
)

DATA_DIR = ROOT / 'demo_data'
DATA_DIR

## 1) Diseño de la base de datos

El modelo separa libros, ediciones, autores, editoriales, ventas y métricas. También incorpora dos tablas de auditoría:

- `source_records`: conserva el valor original, el valor normalizado, la fuente y el estado de cada fila.
- `pipeline_events`: registra cuántas filas entran y salen de cada fase.

La vista `book_summary` reúne la información principal sin desnormalizar las tablas maestras.

In [ ]:
# El esquema completo utilizado por la aplicación.
print(SCHEMA)

## 2) Lectura de fuentes heterogéneas

Cada institución utiliza nombres de columnas distintos para conceptos equivalentes. Antes de concatenar los datos es necesario definir un mapeo explícito.

| Fuente | Archivo demo | Ejemplos de columnas originales |
|---|---|---|
| SIMEH | `simeh.xlsx` | `ISBN`, `Título`, `Autoría`, `Editorial` |
| Plantilla propia | `plantilla_propia.xlsx` | `isbn13`, `title`, `authors`, `publisher` |
| SciELO | `scielo.xlsx` | `ISBN-13`, `TITULO`, `AUTORES`, `EDITOR` |

In [ ]:
source_overview = []
for source_name, spec in SOURCE_SPECS.items():
    raw = pd.read_excel(DATA_DIR / spec['filename'])
    source_overview.append({
        'fuente': source_name,
        'filas': len(raw),
        'columnas': ', '.join(raw.columns),
    })

display(pd.DataFrame(source_overview))

### 2.1) SIMEH

SIMEH actúa como una de las fuentes principales del proyecto original. En la demo se conserva su estructura característica y se incluyen filas de control para comprobar la validación.

In [ ]:
simeh_raw = pd.read_excel(DATA_DIR / 'simeh.xlsx')
display(simeh_raw.head())

### 2.2) Plantilla propia

Esta fuente utiliza nombres de columnas cercanos al modelo canónico. Tiene prioridad durante la deduplicación porque representa los datos ya revisados por el equipo.

In [ ]:
propia_raw = pd.read_excel(DATA_DIR / 'plantilla_propia.xlsx')
display(propia_raw.head())

### 2.3) SciELO

SciELO aporta otra convención de nombres y formatos. El pipeline la adapta sin modificar el archivo de origen.

In [ ]:
scielo_raw = pd.read_excel(DATA_DIR / 'scielo.xlsx')
display(scielo_raw.head())

## 3) Normalización y trazabilidad

Las tres fuentes se transforman a un esquema común. La normalización:

- elimina separadores del ISBN sin perder el valor original;
- crea una versión comparable del título en minúsculas, sin tildes ni puntuación;
- registra el archivo y el número de fila de procedencia;
- conserva simultáneamente campos crudos y normalizados.

In [ ]:
records = _read_sources(DATA_DIR)
display(records[[
    'source_name', 'source_row', 'raw_isbn', 'isbn13',
    'raw_title', 'normalized_title', 'valid_isbn'
]].head(12))

### Ejemplo aislado de las funciones de limpieza

In [ ]:
example = pd.DataFrame({
    'valor_original': ['978-84-204-6472-5', 'Cien años de soledad'],
    'valor_normalizado': [
        clean_isbn('978-84-204-6472-5'),
        normalize_text('Cien años de soledad'),
    ],
})
display(example)

## 4) Validación de ISBN-13

No basta con comprobar que el campo tenga trece dígitos. `is_valid_isbn13` recalcula el dígito de control y rechaza las filas que no superan el checksum. Las filas rechazadas siguen disponibles en la tabla de trazabilidad.

In [ ]:
validation_summary = (
    records.assign(estado=records['valid_isbn'].map({True: 'aceptado', False: 'rechazado'}))
    .groupby(['source_name', 'estado'])
    .size()
    .rename('filas')
    .reset_index()
)
display(validation_summary)
display(records.loc[~records['valid_isbn'], ['source_name', 'source_row', 'raw_isbn', 'raw_title']])

## 5) Deduplicación y catálogo canónico

Un mismo ISBN puede aparecer en varias instituciones. El pipeline agrupa por ISBN-13 válido y aplica una prioridad determinista (`plantilla_propia`, `simeh`, `scielo`). Además, conserva el número y los nombres de las fuentes que respaldan cada edición.

In [ ]:
catalogue = _canonical_catalogue(records)
display(catalogue[[
    'isbn13', 'title', 'authors', 'publisher', 'year',
    'country', 'source_count', 'source_names'
]].sort_values(['source_count', 'title'], ascending=[False, True]))

## 6) APIs e integraciones externas del proyecto original

Las APIs eran una parte relevante del workflow original. Este notebook consulta OpenAlex realmente. Si la red o la API fallan, registra el error y utiliza un fixture claramente identificado para que el resto del ETL pueda terminar.

### 6.1) OpenAlex API

OpenAlex se utilizó para enriquecer los libros con DOI y número de citas. Como la búsqueda de obras no se realizaba directamente por ISBN, el proyecto original implementaba un proceso de selección de candidatos:

1. Obtener el título y los colaboradores desde SQLite.
2. Normalizar mayúsculas, espacios y diacríticos.
3. Consultar `https://api.openalex.org/works` con `type:book`.
4. Comparar cada candidato mediante *fuzzy matching*.
5. Ponderar aproximadamente un 80 % la similitud del título y un 20 % la coincidencia de autores.
6. Guardar el identificador de OpenAlex, el DOI, las citas y el estado de la consulta.

El principal riesgo era asignar un mismo resultado a libros con títulos o subtítulos parecidos. Por eso el original también contemplaba consultas SQL para detectar DOI duplicados asociados a títulos diferentes.

Las filas procedentes de la API quedan marcadas como `openalex_live`. La demo **no presenta las cifras del fallback como datos reales de OpenAlex**: esas filas se identifican como `fixture_fallback`.

In [ ]:
OPENALEX_MODE = 'live'
OPENALEX_API_KEY = os.getenv('OPENALEX_API_KEY')  # Opcional; nunca se guarda en el notebook.
print('Modo OpenAlex:', OPENALEX_MODE)
print('API key configurada:', bool(OPENALEX_API_KEY))

### 6.2) Altmetric

El workflow original exportaba una lista de ISBN y procesaba un CSV de Altmetric para incorporar atención digital. En la demo, `altmetrics.csv` conserva la forma del enriquecimiento —ISBN, menciones, lectores y puntuación—, pero todos sus valores están marcados como sintéticos. No se consulta el servicio externo.

### 6.3) WorldCat Search API

El notebook original incluía un prototipo de consulta de WorldCat por ISBN y región. La integración no llegó a incorporarse al flujo estable porque el acceso a WorldCat Search API requiere suscripción y autenticación.

La versión portfolio mantiene documentada esta decisión técnica, pero no realiza la petición: así evita depender de una cuenta personal o inducir a pensar que la función está disponible sin credenciales.

## 7) Enriquecimiento: ventas y métricas digitales

El proyecto original integraba datos operativos y enriquecimiento externo. Para que la demo sea pública y estable:

- las ventas, ingresos y canales son **sintéticos**;
- las métricas digitales son **sintéticas**;
- OpenAlex se consulta en vivo y su resultado indica siempre la fuente utilizada;
- la clave de OpenAlex es opcional y se lee únicamente desde `OPENALEX_API_KEY`;
- WorldCat y Altmetric no realizan peticiones autenticadas.

In [ ]:
sales = pd.read_excel(DATA_DIR / 'ventas.xlsx')
sales.columns = ['isbn13', 'sale_date', 'units', 'revenue_eur', 'channel', 'synthetic']
altmetrics = pd.read_csv(DATA_DIR / 'altmetrics.csv', dtype={'isbn13': str})

display(sales.groupby('channel', as_index=False).agg(
    unidades=('units', 'sum'),
    ingresos_eur=('revenue_eur', 'sum'),
))
display(altmetrics.head())

## 8) Ejecución completa y persistencia en SQLite

Ahora se ejecuta la misma función que utiliza Streamlit. El resultado incluye la ruta de la base, los dataframes intermedios, el log de eventos y las métricas del proceso.

In [ ]:
database_path = ROOT / 'iberoamerica_books_demo.sqlite'
result = run_pipeline(
    DATA_DIR,
    database_path,
    openalex_mode=OPENALEX_MODE,
    openalex_api_key=OPENALEX_API_KEY,
)

display(pd.DataFrame([result.metrics]).T.rename(columns={0: 'valor'}))
display(result.events)
display(result.openalex[[
    'isbn13', 'query_title', 'match_status', 'data_source',
    'matched_title', 'doi', 'cited_by_count', 'match_score', 'error'
]])
print(f'Base generada en: {result.database_path}')

### Resultado reproducible del dataset demo

| Indicador | Resultado esperado |
|---|---:|
| Filas de entrada | 24 |
| Filas con ISBN inválido | 2 |
| Libros consolidados | 12 |
| Registros de ventas sintéticas | 48 |
| Unidades sintéticas | 1.110 |
| Ingresos sintéticos | 20.265 € |

Estos valores son deterministas y también se verifican mediante tests automáticos.

## 9) Auditoría y control de calidad

La tabla `pipeline_events` permite comprobar la evolución del número de filas. `source_records` permite volver desde un registro final hasta su fila de origen.

In [ ]:
with connect(result.database_path) as connection:
    audit = pd.read_sql_query(
        'SELECT source_name, status, COUNT(*) AS rows '
        'FROM source_records GROUP BY source_name, status '
        'ORDER BY source_name, status',
        connection,
    )
    objects = pd.read_sql_query(
        "SELECT type, name FROM sqlite_master "
        "WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name",
        connection,
    )

display(audit)
display(objects)

## 10) Consultas de ejemplo

La base final puede analizarse directamente con SQL. El helper `read_only_query` limita la demo a instrucciones `SELECT` o `WITH`.

### 10.1) Libros con mayor ingreso sintético

In [ ]:
query = '''
SELECT title, authors, publisher, publication_year, units_sold, revenue_eur
FROM book_summary
ORDER BY revenue_eur DESC
LIMIT 5
'''
with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

### 10.2) Ventas sintéticas por canal

In [ ]:
query = '''
SELECT channel, SUM(units) AS units, ROUND(SUM(revenue_eur), 2) AS revenue_eur
FROM sales
GROUP BY channel
ORDER BY revenue_eur DESC
'''
with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

### 10.3) Procedencia de cada edición

In [ ]:
query = '''
SELECT b.title, e.isbn13, e.source_count, e.source_names
FROM editions AS e
JOIN books AS b ON b.book_id = e.book_id
ORDER BY e.source_count DESC, b.title
'''
with connect(result.database_path) as connection:
    columns, rows = read_only_query(connection, query)
display(pd.DataFrame(rows, columns=columns))

## 11) Diferencias respecto al proyecto original

| Proyecto original | Portfolio / Demo Version |
|---|---|
| Archivos operativos privados | Dataset público semisintético y reducido |
| Integraciones externas variables | OpenAlex live con fallback explícito |
| Notebook exploratorio de gran tamaño | Pipeline modular compartido con Streamlit |
| Resultados y outputs operativos | Métricas de ejemplo sin información sensible |

La demo conserva el problema técnico, el modelo relacional, las reglas de limpieza, la trazabilidad y el patrón de consultas; no pretende reproducir el volumen ni las conclusiones comerciales de los datos privados.

## 12) Limitaciones

- La escala se ha reducido para que el workflow pueda revisarse rápidamente.
- Las cifras comerciales y de atención digital no representan actividad real.
- Los resultados live de OpenAlex pueden cambiar con el tiempo; el modo offline sigue disponible para ejecuciones deterministas.
- La calidad final de cualquier integración depende de la calidad y consistencia de sus fuentes.

**Demo interactiva:** [iberoamericabooks-etl.streamlit.app](https://iberoamericabooks-etl.streamlit.app/)  
**Código fuente:** [github.com/DiegoJSN/iberoamericabooks_demo](https://github.com/DiegoJSN/iberoamericabooks_demo)